# 9. Lakebase - Fully Managed Postgres OLTP Database

Lakebase is Databricks' fully managed Postgres online transaction processing (OLTP) database engine integrated directly into the Data Intelligence Platform. Build and operate transactional databases with the simplicity of Databricks-managed infrastructure while maintaining seamless integration with your lakehouse ecosystem.

## Lakebase Overview
[![Video Thumbnail](https://img.youtube.com/vi/3Bmnku-x0Yo/0.jpg)](https://www.youtube.com/watch?v=3Bmnku-x0Yo)

## What is Lakebase?

**Lakebase** is a serverless, fully managed PostgreSQL-compatible database designed for real-time transactional workloads within the Databricks platform.

### Key Benefits:
* **Fully Managed** - Databricks handles infrastructure, maintenance, patching, and scaling
* **PostgreSQL Compatible** - Use standard Postgres SQL, tools, and drivers
* **Lakehouse Integration** - Seamlessly connects with Unity Catalog tables and data workflows
* **Real-Time Transactions** - ACID compliance for mission-critical transactional workloads
* **AI-Native** - Built for modern AI applications and data apps
* **Database Branching** - Create point-in-time copies for development, testing, and compliance

### Common Use Cases:
* Transactional backends for web and mobile applications
* Real-time data apps requiring low-latency reads and writes
* Development and testing environments with database branching
* Operational databases for AI agents and chatbots
* Compliance and auditing with point-in-time recovery
* Microservices requiring independent operational datastores

**Note:** Lakebase is in public preview in the following regions: us-east-1, us-west-2, eu-west-1, ap-southeast-1, ap-southeast-2, eu-central-1, us-east-2, ap-south-1

📖 **Resource:** [Lakebase Documentation](https://docs.databricks.com/en/oltp/index.html)

## Method 1: Creating a Lakebase Instance from the UI

The easiest way to get started with Lakebase is through the Databricks workspace UI.

### To Create Your First Instance:
1. In your Databricks workspace, navigate to **Compute** in the left sidebar
2. Click on the **Lakebase Postgres** 
3. Click **Create database instance**
4. Enter a database instance name 
5. Choose compute size (CU_1, CU_2, etc.) based on workload
6. (Optional) Configure advanced settings:
7. Click **Create** to provision your instance

Your Lakebase instance will be ready in a few minutes!

📖 **Resource:** [Create and manage a database instance](https://docs.databricks.com/aws/en/oltp/instances/create/)

## Method 2: Creating an Instance with Python SDK

For programmatic control and automation, use the Databricks SDK to create and manage Lakebase instances.

Below is an example of creating a Lakebase instance using Python:

In [ ]:
# Example: Create a Lakebase instance using the Databricks SDK
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.database import DatabaseInstance

# Initialize the Databricks workspace client
w = WorkspaceClient()

# Create a new Lakebase database instance
instance = w.database.create_database_instance(
  DatabaseInstance(
    name="dnam-test-instance",
    capacity="CU_2",  # Compute capacity: CU_1, CU_2, CU_4, etc.
  )
)

print(f"Instance created: {instance.name}")
print(f"Instance ID: {instance.id}")
print(f"Status: {instance.state}")

## Register Lakebase Database to Unity Catalog

Once you create the lakebase instance, you'll have to register it as a catalog in Unity Catalog. 

You can do this by via the UI:
1. Click Compute in the workspace sidebar.
2. On the Database instances tab, select your database instance.
3. On the Catalogs tab, click Create catalog.
4. You can register an existing database as a Unity Catalog catalog or create a new one. Use one of the following options:
- To use an existing database: enter the desired Unity Catalog catalog name and the name of the existing Postgres database. The default database databricks_postgres can be used as the database name.
- To create a new Postgres database and Unity Catalog catalog at the same time: enter the desired catalog name and turn on Create new database.
5. Click Create.

After creation, click on the catalog in the Catalogs list to see the Catalog Explorer view.

Or using the SDK. Below is an example of registering a Lakebase instance to Unity Catalog using Python:

📖 **Resource:** [Register Database to Unity Catalog](https://docs.databricks.com/aws/en/oltp/instances/register-uc)

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.database import DatabaseCatalog

# Initialize the Workspace client
w = WorkspaceClient()

# Register an existing database as a UC catalog
catalog = w.database.create_database_catalog(
    DatabaseCatalog(
        name="my_catalog",                    # Name of the UC catalog to create
        database_instance_name="my-instance", # Name of the database instance
        database_name="databricks_postgres",  # Name of the existing Postgres database
    )
)
print(f"Created database catalog: {catalog.name}")

# Create a new database and register it as a UC catalog
catalog = w.database.create_database_catalog(
    DatabaseCatalog(
        name="new_catalog",                   # Name of the UC catalog to create
        database_instance_name="my-instance", # Name of the database instance
        database_name="new_database",         # Name of the Postgres database to register (and optionally create)
        create_database_if_not_exists=True    # Create the database if it doesn't exist
    )
)
print(f"Created new database and catalog: {catalog.name}")

## Connecting to Lakebase

Once your Lakebase instance is created, you can connect to it using standard PostgreSQL connection methods.

### Getting Connection Details

1. Navigate to **Compute** → **OLTP Database**
2. Click on your instance name
3. Click **Connection details** to see:
   - Host name (e.g., `instance-123abc.database.cloud.databricks.com`)
   - User
   - Port (typically `5432`)
   - Database name
   - sslmode

Create OAuth token for CLI connection to the database.

### Connect Using psql Command Line

Connect using the standard PostgreSQL command-line client:

```bash
psql "host=instance-123abc.database.cloud.databricks.com user=user@databricks.com dbname=databricks_postgres port=5432 sslmode=require"
```

You are also able to connect to the database using the Databricks python SDK. For details, refer to the resource below:

📖 **Resource:** [Authenticate to a database instance](https://docs.databricks.com/aws/en/oltp/instances/authentication?language=Python+SDK)

You are also able to connect and query your database from SQL Editor, Databricks notebooks, and SQL Clients. For detials, refer to the resource below:

📖 **Resource:** [Authenticate to a database instance](https://docs.databricks.com/aws/en/oltp/instances/query/)

You can also refer to code examples using psycopg2, psycopg3, SQL Alchemy, and Scala from the [Query from notebooks documentation](https://docs.databricks.com/aws/en/oltp/instances/query/notebook#gsc.tab=0)

## Sync data from Unity Catalog to Lakebase

One of Lakebase's powerful features is its integration with Unity Catalog, allowing you to create and manage synced tables. A synced table is a Unity Catalog read-only Postgres table that automatically synchronizes data from a Unity Catalog table to your Lakebase database instance. Syncing a Unity Catalog table to Postgres enables low-latency queries and query-time joins with other Postgres tables.

Lakeflow Declarative Pipelines manages the synchronization, continuously propagating changes from the source table to Postgres. Once created, synced tables can be queried directly using standard Postgres tools.

Key characteristics of synced tables:

- Read-only in Postgres to maintain data integrity with the source
- Automatically synchronized using managed Lakeflow Declarative Pipelines
- Queryable through standard PostgreSQL interfaces
- Managed through Unity Catalog for governance and lifecycle management

### Create a synced table 

You can try this with `samples.nyctaxi.trips`

1. Click Catalog in the workspace sidebar.
2. Find and select the Unity Catalog table you want to create a synced table on.
3. Click Create > Synced table.
4. Select your catalog, schema, and enter a table name for the new synced table.
- Synced tables can also be created in Standard catalogs, with some additional configuration. Select your Standard catalog, a schema, and enter a table name for the newly created synced table.
5. Select a database instance and enter the name of the Postgres database in which to create the synced table. The Postgres database field defaults to the currently selected target catalog. If a Postgres database does not exist under this name, Databricks creates a new one.
6. Select a Primary Key. A primary key is required as it enables efficient access to rows for reads, updates, and deletes.
7. If two rows have the same primary key in the source table, select a Timeseries Key to configure deduplication. When a Timeseries Key is specified, the synced tables contains only the rows with the latest timeseries key value for each primary key.
8. Select the sync mode from **Snapshot, Triggered, and Continuous**. For more information about each sync mode, see Sync modes explained.
9. Choose if you want to create this synced table from a new or existing pipeline.
- If creating a new pipeline and using a managed catalog, choose the storage location for the staging table. If using a standard catalog, the staging table is automatically stored in the catalog.
- If using an existing pipeline, check that the new sync mode matches the pipeline mode.
10. (Optional) Select a Serverless budget policy. To create a serverless budget policy, see Attribute usage with serverless budget policies. This allows you to attribute billing usage to specific usage policies.
For synced tables, the billable entity is the underlying Lakeflow Declarative Pipelines pipeline. To modify the budget policy, modify the underlying pipeline object. See Configure a serverless pipeline.
11. After Synced table status is Online, log in to your database instance and query the newly created table. Query your table using the SQL editor, external tools, or notebooks.

You are also able to create synced using the Databricks python SDK. For details, refer to the resource below:

📖 **Resource:** [Sync data from UC to database](https://docs.databricks.com/aws/en/oltp/instances/sync-data/sync-table?language=Python+SDK)

## Check In

At this point, you should have a Lakebase instance running with a synced table set up. You should be able to connect to the postgres instance using the CLI to verify that the UC table has been successfully synced with the database. 

Please refer to the [Databricks Lakebase Documentation](https://docs.databricks.com/aws/en/oltp/) for additional features you are interested in testing.

## Database Branching

Lakebase supports **database branching**, allowing you to create point-in-time copies of your database.

### Creating a Branch:

1. Navigate to **Compute** → **Lakebase Postgres**
2. Click **Create database instance**
3. In **Advanced Settings**, enable **Create from parent**
4. Select the parent instance and choose a point in time
5. Click **Create** to provision the child instance

📖 **Resource:** [Create a child instance](https://docs.databricks.com/aws/en/oltp/instances/create/child-instance)

## Metrics and Performance

Lakebase provides built-in monitoring capabilities to track database health and performance:

### Viewing Instance Metrics:

1. Navigate to **Compute** → **Lakebase Postgres**
2. Click on your instance name
3. View the **Metrics** tab for:
   - Transactions per second
   - Storage Utilization
   - CPU Utilization
   - Rows per second

and more...

📖 **Resource:** [Monitor a database instance](https://docs.databricks.com/aws/en/oltp/instances/create/monitor)

## High Availability and Disaster Recovery

Lakebase provides enterprise-grade reliability features:

### High Availability:
To enable high availability, specify additional nodes as part of a database instance. If the primary compute becomes unhealthy or unavailable, a high availability node is utilized to perform failover, and the secondary node is promoted to primary.

### Point-in-Time Recovery:
* Configure retention windows (1-35 days)
* Restore to any point within the retention period
* Create child instances from specific timestamps

### Backup Strategy:
* Automated continuous backups
* No performance impact during backup
* Quick restore capabilities

📖 **Resource:** [High availability configuration](https://docs.databricks.com/en/oltp/high-availability.html)

## Integration with Databricks Apps

Lakebase works seamlessly with Databricks Apps for building interactive applications. A complete Streamlit Todo App example using Lakebase as the backend database is available in notebook **[8. Databricks Apps.ipynb](8.%20Databricks%20Apps.ipynb)** (cell 12).

📖 **Resource:** [Add a Lakebase resource to Databricks app](https://docs.databricks.com/aws/en/dev-tools/databricks-apps/lakebase)

### Key Implementation Patterns

The Todo App demonstrates production-ready patterns for connecting Databricks Apps to Lakebase:

#### 1. OAuth Token Management with Auto-Refresh
```python
from databricks import sdk

workspace_client = sdk.WorkspaceClient()
postgres_password = None
last_password_refresh = 0

def refresh_oauth_token():
    """Refresh OAuth token if expired (recommended every 15 minutes)"""
    global postgres_password, last_password_refresh
    if postgres_password is None or time.time() - last_password_refresh > 900:
        postgres_password = workspace_client.config.oauth_token().access_token
        last_password_refresh = time.time()
```

**Why this matters:** OAuth tokens expire, so automatic refresh ensures your app maintains connectivity without manual intervention.

#### 2. Connection Pooling for Performance
```python
from psycopg_pool import ConnectionPool

def get_connection_pool():
    """Create connection pool with automatic token refresh"""
    global connection_pool
    if connection_pool is None:
        refresh_oauth_token()
        conn_string = (
            f"dbname={os.getenv('PGDATABASE')} "
            f"user={os.getenv('PGUSER')} "
            f"password={postgres_password} "
            f"host={os.getenv('PGHOST')} "
            f"port={os.getenv('PGPORT')} "
            f"sslmode={os.getenv('PGSSLMODE', 'require')}"
        )
        connection_pool = ConnectionPool(conn_string, min_size=2, max_size=10)
    return connection_pool
```

**Why this matters:** Connection pools reuse database connections, dramatically improving performance and reducing overhead compared to creating new connections for each query.

#### 3. Dynamic Schema Management per App/User
```python
def get_schema_name():
    """Generate unique schema name for isolation"""
    pgappname = os.getenv("PGAPPNAME", "my_app")
    pguser = os.getenv("PGUSER", "").replace('-', '')
    return f"{pgappname}_schema_{pguser}"
```

**Why this matters:** Each app or user gets their own schema, providing data isolation and multi-tenancy support within a single database instance.

#### 4. Automatic Database Initialization
```python
def init_database():
    """Initialize schema and tables on first run"""
    with get_connection() as conn:
        with conn.cursor() as cur:
            schema_name = get_schema_name()
            cur.execute(sql.SQL("CREATE SCHEMA IF NOT EXISTS {}").format(
                sql.Identifier(schema_name)
            ))
            cur.execute(sql.SQL("""
                CREATE TABLE IF NOT EXISTS {}.todos (
                    id SERIAL PRIMARY KEY,
                    task TEXT NOT NULL,
                    completed BOOLEAN DEFAULT FALSE,
                    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                )
            """).format(sql.Identifier(schema_name)))
            conn.commit()
```

**Why this matters:** Apps automatically set up required database structures on first launch, eliminating manual setup steps.

#### 5. Real-Time CRUD Operations
The Todo App implements full CRUD (Create, Read, Update, Delete) operations with Streamlit's fragment feature for efficient UI updates:

```python
@st.fragment
def display_todos():
    todos = get_todos()
    for todo_id, task, completed, created_at in todos:
        # Interactive UI with real-time updates
        if st.checkbox("", value=completed, key=f"check_{todo_id}"):
            toggle_todo(todo_id)
            st.rerun(scope="fragment")  # Efficient partial re-render
```

**Why this matters:** Fragment-based updates only re-render affected UI components, providing a smooth user experience without full page reloads.

### Full Working Example

See **[8. Databricks Apps.ipynb](8.%20Databricks%20Apps.ipynb)** cell 12 for the complete template available in the Databricks Apps UI.

## Comprehensive Resource Library

### 📚 **Official Documentation**
* [Lakebase - Complete Guide](https://docs.databricks.com/en/oltp/index.html)
* [Create and manage database instances](https://docs.databricks.com/aws/en/oltp/instances/create/)
* [Connect to Lakebase](https://docs.databricks.com/aws/en/oltp/instances/authentication)
* [Database branching](https://docs.databricks.com/aws/en/oltp/instances/create/child-instance)
* [Sync data from Unity Catalog](https://docs.databricks.com/aws/en/oltp/instances/sync-data/sync-table)
* [High availability configuration](https://docs.databricks.com/aws/en/oltp/instances/create/high-availability)
* [PostgreSQL compatibility](https://docs.databricks.com/aws/en/oltp/instances/query/postgres-compatibility)

### 🛠️ **Tools and Libraries**
* [psycopg - PostgreSQL adapter for Python](https://www.psycopg.org/psycopg3/)
* [Databricks SDK for Python](https://docs.databricks.com/aws/en/dev-tools/sdk-python)
* [PostgreSQL Documentation](https://www.postgresql.org/docs/)

### 🎓 **Additional Resources**
* [How to use Lakebase for Apps - Blog Post](https://www.databricks.com/blog/how-use-lakebase-transactional-data-layer-databricks-apps?utm_source=bambu&utm_medium=social)
* [Building Apps with Lakebase](https://docs.databricks.com/en/dev-tools/databricks-apps/index.html)
* [dbt with Lakebase](https://docs.getdbt.com/docs/core/connect-data-platform/lakebase-setup)